# Example queries: `savings` (comstock_oedi_agg)

Auto-generated from `tests/query_snapshots/savings.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from pathlib import Path
from buildstock_query import BuildStockQuery
from buildstock_query.schema.utilities import MappedColumn
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [ ]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/comstock_oedi_agg_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/comstock_oedi_agg_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_agg_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


## `savings_annual_upgrade1_electricity`

Annual savings for upgrade 1, electricity, grouped by building type, CO only.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    upgrade_id='1',
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    include_baseline=True,
    include_upgrade=True,
    include_savings=True,
)
result.head() if hasattr(result, 'head') else result


## `savings_ts_monthly_upgrade1_electricity`

Monthly timeseries savings for upgrade 1, electricity, CO only.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    upgrade_id='1',
    annual_only=False,
    timestamp_grouping_func='month',
    group_by=['time'],
    restrict=[('state', ['CO'])],
    include_baseline=True,
    include_upgrade=True,
    include_savings=True,
)
result.head() if hasattr(result, 'head') else result


## `savings_only_upgrade1_electricity`

Savings-only annual output (no baseline / no upgrade columns) for upgrade 1, CO. Catches column-suppression bugs in the include_baseline / include_upgrade flags.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    upgrade_id='1',
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    include_baseline=False,
    include_upgrade=False,
    include_savings=True,
)
result.head() if hasattr(result, 'head') else result


## `savings_annual_upgrade1_quartiles`

Annual upgrade 1 savings with quartiles, CO only.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    upgrade_id='1',
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    include_baseline=True,
    include_upgrade=True,
    include_savings=True,
    get_quartiles=True,
)
result.head() if hasattr(result, 'head') else result


## `savings_annual_two_fuel`

Annual savings for upgrade 1 covering both electricity and natural gas as enduses, grouped by building type, CO only. Pins the multi-fuel column-suffix shape across the baseline/upgrade/savings projection — every prior savings entry is electricity-only.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh', 'out.natural_gas.total.energy_consumption..kwh'],
    upgrade_id='1',
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    include_baseline=True,
    include_upgrade=True,
    include_savings=True,
)
result.head() if hasattr(result, 'head') else result


## `savings_ts_monthly_two_fuel`

Monthly TS savings for upgrade 1 covering both electricity and natural gas, grouped by time, CO only. Mirrors savings_annual_two_fuel on the TS path — pins the multi-fuel column-suffix shape across baseline/upgrade/savings projections in the TS upgrade-pair flow. Comstock skips per UnsupportedQueryShape (TS upgrade-pair queries on comstock).


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption', 'out.natural_gas.total.energy_consumption'],
    upgrade_id='1',
    annual_only=False,
    timestamp_grouping_func='month',
    group_by=['time'],
    restrict=[('state', ['CO'])],
    include_baseline=True,
    include_upgrade=True,
    include_savings=True,
)
result.head() if hasattr(result, 'head') else result
